# Skills y gestión de skills

**Lección 2 · Clase 5.4** — la lección 1 conectó al agente con *herramientas*: cosas que puede hacer. Esta lo conecta con **conocimiento operativo**: cómo se hacen las cosas *acá*. El formato del informe diario, el protocolo de reclamos, las reglas del aviso de cierre — nada de eso es una herramienta, y meterlo todo en el system prompt no escala.

Una **skill** es conocimiento procedural empaquetado: una carpeta con un `SKILL.md` — metadatos arriba, instrucciones abajo — que el agente carga *solo cuando la tarea lo amerita*. Y desde diciembre de 2025 es un **estándar abierto** ([agentskills.io](https://agentskills.io)): el mismo archivo lo entienden Claude Code, Codex CLI, GitHub Copilot, Cursor, Gemini CLI y decenas de herramientas más. Escribes el conocimiento una vez; cualquier agente lo usa.

> Esta lección se enfoca en el **estándar, el mecanismo por dentro y la distribución multi-agente**. El *versionado* de skills (v1→v2, packages, lockfile) ya lo cubre la [lección 3 de la clase 3.6](../../class_3_6_production/leccion3_versionamiento_skills/) — acá no lo repetimos.

| | |
|---|---|
| **Anatomía** | El SKILL.md por dentro, y por qué la `description` es la pieza crítica. |
| **El mecanismo pelado** | Un skill-loader de ~30 líneas sobre la API cruda de OpenAI: *progressive disclosure* desmitificado. |
| **El contraste** | La misma tarea con y sin skill — y la cuenta de tokens que explica por qué el patrón escala. |
| **Gestión en equipo** | Packmind: la misma skill materializada para tres agentes distintos con un comando. |
| **El fallback** | Un `distribuir_skill()` casero que enseña qué hace Packmind por dentro. |


In [ ]:
# Esta lección usa el entorno uv del README. Si la corres en Colab, descomenta:
# %pip install -q openai==2.53.0 pyyaml==6.0.3 python-dotenv==1.2.2
from dotenv import load_dotenv
import os
import shutil

load_dotenv(override=True)  # el .env de la lección gana sobre variables heredadas del entorno

try:
    from google.colab import userdata  # type: ignore
    for llave in ("OPENAI_API_KEY", "PACKMIND_API_KEY_V3"):
        try:
            os.environ[llave] = userdata.get(llave) or os.environ.get(llave, "")
        except Exception:
            pass
except Exception:
    pass

HAY_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
HAY_PACKMIND = bool(os.environ.get("PACKMIND_API_KEY_V3")) and shutil.which("npx") is not None

print("OPENAI_API_KEY presente:", HAY_OPENAI)
print("Packmind disponible (llave + npx):", HAY_PACKMIND)
if not HAY_OPENAI:
    print("⚠️ Sin OPENAI_API_KEY el loader y el contraste se saltan.")
if not HAY_PACKMIND:
    print("⚠️ Sin Packmind, la sección de distribución usa solo el fallback local")
    print("   (cuenta gratis en app.packmind.com; la CLI necesita Node ≥ 20).")

# En Colab las skills no existen localmente: bajarlas del repo público.
import urllib.request
from pathlib import Path

BASE_RAW = (
    "https://raw.githubusercontent.com/josepenam/clases-diplomado-gen-ia/main/"
    "class_5_4_integraciones/leccion2_skills_y_gestion/"
)


def asegurar(nombre: str) -> Path:
    ruta = Path(nombre)
    if not ruta.exists():
        ruta.parent.mkdir(parents=True, exist_ok=True)
        pedido = urllib.request.Request(
            BASE_RAW + nombre, headers={"User-Agent": "Mozilla/5.0 (clase-diplomado-gen-ia)"}
        )
        ruta.write_bytes(urllib.request.urlopen(pedido).read())
    return ruta


for skill in ("informe-nieve", "aviso-cierre-pistas", "respuesta-reclamos"):
    asegurar(f"skills/{skill}/SKILL.md")
print("✓ skills disponibles en ./skills/")

## Anatomía de una skill

Dos partes, y la separación es el corazón del estándar:

- **Frontmatter** (`name`, `description`): lo único que el agente ve *siempre*. Barato — un par de líneas por skill en el contexto.
- **Cuerpo**: las instrucciones completas. Se cargan **solo cuando la tarea lo amerita**.

Eso es *progressive disclosure*, y hace que la `description` sea la pieza crítica de todo el formato: es lo único que el modelo tiene para decidir si esta skill aplica. Una buena description dice **qué hace Y cuándo usarla** ("Usar cuando pidan un informe, parte o reporte de nieve") — una mala solo describe el tema y el agente nunca la carga.

In [ ]:
import yaml

DIR_SKILLS = Path("skills")


def leer_skill(ruta_md: Path) -> dict:
    """Separa frontmatter y cuerpo de un SKILL.md."""
    _, frontmatter, cuerpo = ruta_md.read_text().split("---", 2)
    return {**yaml.safe_load(frontmatter), "cuerpo": cuerpo.strip(), "ruta": ruta_md}


def descubrir_skills(directorio: Path = DIR_SKILLS) -> list[dict]:
    return [leer_skill(p) for p in sorted(directorio.glob("*/SKILL.md"))]


skills = descubrir_skills()
print(f"{len(skills)} skills descubiertas:\n")
for s in skills:
    print(f"  {s['name']:<22} {s['description'][:80]}…")

print("\n── el SKILL.md completo de la estrella de la lección ──\n")
print(Path("skills/informe-nieve/SKILL.md").read_text())

## El mecanismo pelado: un skill-loader en ~30 líneas

Antes de usar skills a través de un framework, veamos qué son *por dentro*. Sin LangChain, sin SDK de agentes: la API cruda de OpenAI y un loop de tool-calling manual.

El truco completo cabe en tres pasos:

1. El **catálogo** (solo names + descriptions) va en el system prompt.
2. Le damos al modelo una única herramienta: `cargar_skill(nombre)`.
3. Si la llama, le devolvemos el cuerpo del SKILL.md como resultado — **inyección condicional de contexto**. Eso es todo. No hay magia.

Cualquier producto que "soporta skills" —Claude Code, Codex, Cursor— implementa alguna variante de esto.

In [ ]:
import json

from openai import OpenAI

MODELO = "gpt-5-mini"

CATALOGO = "\n".join(f"- {s['name']}: {s['description']}" for s in skills)

SISTEMA_CON_SKILLS = f"""Eres el asistente de operaciones de un centro de esquí chileno.

Tienes disponibles estas skills (conocimiento procedural del equipo):
{CATALOGO}

Si la tarea calza con una skill, cárgala con la herramienta cargar_skill ANTES
de responder, y sigue sus instrucciones al pie de la letra."""

HERRAMIENTA_CARGAR = {
    "type": "function",
    "function": {
        "name": "cargar_skill",
        "description": "Devuelve las instrucciones completas de una skill del catálogo.",
        "parameters": {
            "type": "object",
            "properties": {"nombre": {"type": "string", "description": "El name de la skill"}},
            "required": ["nombre"],
        },
    },
}


def agente_con_skills(tarea: str) -> tuple[str, list[str], int]:
    """Loop manual de tool-calling. Devuelve (respuesta, skills_cargadas, prompt_tokens)."""
    cliente = OpenAI()
    mensajes = [{"role": "system", "content": SISTEMA_CON_SKILLS},
                {"role": "user", "content": tarea}]
    cargadas, tokens_prompt = [], 0

    for _ in range(4):  # a lo más 4 vueltas
        respuesta = cliente.chat.completions.create(
            model=MODELO, messages=mensajes, tools=[HERRAMIENTA_CARGAR]
        )
        tokens_prompt += respuesta.usage.prompt_tokens
        mensaje = respuesta.choices[0].message
        if not mensaje.tool_calls:
            return mensaje.content, cargadas, tokens_prompt

        mensajes.append(mensaje)
        for llamada in mensaje.tool_calls:
            nombre = json.loads(llamada.function.arguments)["nombre"]
            skill = next((s for s in skills if s["name"] == nombre), None)
            cuerpo = skill["cuerpo"] if skill else f"(no existe la skill {nombre})"
            cargadas.append(nombre)
            mensajes.append({"role": "tool", "tool_call_id": llamada.id, "content": cuerpo})

    return "(el agente no terminó en 4 vueltas)", cargadas, tokens_prompt


print("Skill-loader listo: catálogo de", len(skills), "skills +", "1 herramienta.")

## El contraste: la misma tarea, con y sin

La tarea del día: redactar el parte de nieve con los datos de esta mañana. El agente *sin* skills sabe escribir — pero no sabe **cómo se escribe acá**: el formato oficial, el orden de los sectores, la línea de cierre obligatoria. Verificamos eso último programáticamente, porque es lo que un canal oficial validaría.

In [ ]:
DATOS_HOY = """Datos de hoy (2026-07-15):
- La Plaza (cota 2860, verde): base 155 cm, cayeron 12 cm, riesgo bajo
- El Andino (cota 3100, azul): base 180 cm, cayeron 15 cm, riesgo moderado
- Tres Puntas (cota 3320, roja): base 205 cm, cayeron 18 cm, riesgo moderado
- Barros Negros (cota 3670, negra): base 240 cm, cayeron 25 cm, riesgo alto"""

TAREA = "Redacta el parte de nieve de hoy.\n\n" + DATOS_HOY
LINEA_OFICIAL = "Emitido por Operaciones de Montaña"

if HAY_OPENAI:
    cliente = OpenAI()

    # (a) sin skills: el modelo, solo con su buen gusto
    sin = cliente.chat.completions.create(
        model=MODELO,
        messages=[{"role": "system", "content": "Eres el asistente de operaciones de un centro de esquí chileno."},
                  {"role": "user", "content": TAREA}],
    )
    respuesta_sin = sin.choices[0].message.content

    # (b) con el skill-loader
    respuesta_con, cargadas, tokens_con = agente_con_skills(TAREA)

    for etiqueta, texto in (("SIN skills", respuesta_sin), ("CON skills", respuesta_con)):
        cumple = "✓" if LINEA_OFICIAL in texto else "✗"
        print(f"═══ {etiqueta} — ¿línea oficial de cierre? {cumple} ═══")
        print(texto[:600])
        print("…\n" if len(texto) > 600 else "")
    print("Skills cargadas por el loader:", cargadas)
else:
    print("⛔ Falta OPENAI_API_KEY.")

El agente sin skills produce un parte *razonable* — y inútil para el canal oficial: formato inventado, sin la línea de cierre que valida el sistema. El agente con skills cargó `informe-nieve` (y solo esa: fíjate que no cargó las de reclamos ni cierres) y siguió el formato.

## La cuenta de tokens: por qué escala

¿Y si en vez del catálogo metemos **todas las skills completas** en el system prompt? Con 3 skills, da casi lo mismo. El punto es la pendiente:

In [ ]:
if HAY_OPENAI:
    SISTEMA_TODO_ADENTRO = (
        "Eres el asistente de operaciones de un centro de esquí chileno.\n\n"
        + "\n\n".join(f"## Skill: {s['name']}\n{s['cuerpo']}" for s in skills)
    )
    todo = cliente.chat.completions.create(
        model=MODELO,
        messages=[{"role": "system", "content": SISTEMA_TODO_ADENTRO},
                  {"role": "user", "content": TAREA}],
    )

    chars_catalogo = len(CATALOGO)
    chars_cuerpos = sum(len(s["cuerpo"]) for s in skills)

    print(f"{'estrategia':<38} {'prompt_tokens':>13}")
    print("-" * 53)
    print(f"{'todo adentro (3 skills completas)':<38} {todo.usage.prompt_tokens:>13}")
    print(f"{'progressive disclosure (loader)':<38} {tokens_con:>13}")
    print()
    print(f"El catálogo pesa {chars_catalogo} caracteres; los cuerpos, {chars_cuerpos}.")
    print(f"Con {len(skills)} skills, 'todo adentro' incluso GANA: el loader paga dos llamadas.")
    print("Lo que importa es la pendiente: 'todo adentro' crece con el total de los")
    print("cuerpos (hagan falta o no); el loader crece con el catálogo — una línea")
    print("por skill — más solo el cuerpo que la tarea amerita. Con las ~50 skills")
    print("de un equipo real, la diferencia es un orden de magnitud por llamada.")

## Gestión en equipo: Packmind

El estándar resuelve el *formato*. Queda el problema organizacional: el equipo tiene 15 repos, cada dev usa un agente distinto (Claude Code, Cursor, Copilot...), ¿y la skill del informe de nieve vive... copiada y pegada en cada repo, desactualizándose en cada copia?

[Packmind](https://packmind.com) ataca eso: un **playbook central** (open source, con tier cloud gratis o self-hosted) donde las skills se publican, se empaquetan y se **distribuyen** a cada repo *renderizadas para el agente que use cada quien*. La demo mínima: publicar nuestra skill y verla materializarse para tres agentes distintos con un `install`.

In [ ]:
import re
import subprocess
import sys

BASE_DIR = Path.cwd()
SKILL_REPO = BASE_DIR / "skill_repo"                  # donde AUTORAMOS y publicamos
CONSUMIDOR = BASE_DIR / "proyecto_consumidor"         # un repo cualquiera del equipo

ANSI = re.compile(r"\x1b\[[0-9;]*m")


def packmind(*args, cwd=BASE_DIR, check=True, stdin_text=None, retries=0, silencio=False):
    """Ejecuta `npx @packmind/cli <args>`. Con reintentos: el server a veces
    devuelve un 500 transitorio y nuestras operaciones son idempotentes."""
    kwargs = dict(cwd=str(cwd), capture_output=True, text=True, env=os.environ.copy())
    if stdin_text is not None:
        kwargs["input"] = stdin_text
    else:
        kwargs["stdin"] = subprocess.DEVNULL
    intento = 0
    while True:
        resultado = subprocess.run(["npx", "--yes", "@packmind/cli", *args], **kwargs)
        if resultado.returncode == 0 or intento >= retries:
            if not silencio:
                print(ANSI.sub("", (resultado.stdout or "") + (resultado.stderr or "")))
            if check and resultado.returncode != 0:
                raise RuntimeError(f"packmind {' '.join(args)} devolvió {resultado.returncode}")
            return resultado
        intento += 1
        print(f"[reintento {intento}/{retries}] packmind {args[0]} falló…")


if HAY_PACKMIND:
    packmind("whoami", check=False)
else:
    print("⛔ Sin Packmind: salta a la sección del fallback local, que enseña lo mismo por dentro.")

**Autoría y publicación.** Inicializamos el repo de autoría eligiendo **tres agentes destino** — AGENTS.md (el genérico), Claude Code y Cursor — copiamos la skill al lugar que la CLI espera, y la publicamos al playbook central:

In [ ]:
if HAY_PACKMIND:
    if SKILL_REPO.exists():
        shutil.rmtree(SKILL_REPO)
    SKILL_REPO.mkdir(parents=True)

    # el menú de init es multi-select: 1=AGENTS.md, 2=Claude Code, 6=Cursor
    packmind("init", cwd=SKILL_REPO, stdin_text="1,2,6\n", check=False, retries=2)

    destino = SKILL_REPO / ".claude" / "skills" / "informe-nieve" / "SKILL.md"
    destino.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy("skills/informe-nieve/SKILL.md", destino)

    packmind("playbook", "add", str(destino.parent.relative_to(SKILL_REPO)),
             cwd=SKILL_REPO, check=False, retries=3)
    packmind("playbook", "submit", "--no-review", "-m", "informe-nieve v1 (clase 5.4)",
             cwd=SKILL_REPO, check=False, retries=3)

**Empaquetado.** Un *package* agrupa artefactos instalables (el slug exacto lo leemos de la salida de la CLI, no lo adivinamos):

In [ ]:
PACKAGE_SLUG = None

if HAY_PACKMIND:
    NOMBRE_PACKAGE = "Skills Clase 5.4"
    creado = packmind("packages", "create", NOMBRE_PACKAGE, "-d",
                      "Skills de la clase de integraciones", cwd=SKILL_REPO, check=False, silencio=True)
    listado = packmind("packages", "list", cwd=SKILL_REPO, check=False, silencio=True)

    for texto in (creado.stdout, creado.stderr, listado.stdout):
        if m := re.search(r"@[\w.-]+/[\w.-]*clase[\w.-]*", ANSI.sub("", texto or "")):
            PACKAGE_SLUG = m.group(0)
            break
    print("PACKAGE_SLUG =", PACKAGE_SLUG)

    if PACKAGE_SLUG:
        packmind("packages", "add", "--to", PACKAGE_SLUG, "--skill", "informe-nieve",
                 cwd=SKILL_REPO, check=False, retries=2)

**El payoff.** Ahora somos *otro* proyecto del equipo — uno cualquiera — donde conviven usuarios de Claude Code y de Cursor. Un `install` y la misma skill se materializa **en la carpeta nativa de cada agente**:

In [ ]:
if HAY_PACKMIND and PACKAGE_SLUG:
    if CONSUMIDOR.exists():
        shutil.rmtree(CONSUMIDOR)
    CONSUMIDOR.mkdir(parents=True)

    packmind("init", cwd=CONSUMIDOR, stdin_text="1,2,6\n", check=False, retries=2, silencio=True)
    packmind("install", PACKAGE_SLUG, cwd=CONSUMIDOR, check=False, retries=2)

    print("\n── lo que quedó materializado en el proyecto consumidor ──\n")
    archivos = sorted(p for p in CONSUMIDOR.rglob("*") if p.is_file() and "node_modules" not in p.parts)
    for archivo in archivos:
        print(f"  {archivo.relative_to(CONSUMIDOR)}")

    for candidato in archivos:
        rel = str(candidato.relative_to(CONSUMIDOR))
        if "informe-nieve" in rel or rel == "AGENTS.md":
            print(f"\n═══ {rel} (primeras líneas) ═══")
            print("\n".join(candidato.read_text().splitlines()[:8]))

Una fuente de verdad, una materialización por agente. Fíjate en el detalle lindo: Claude Code y Cursor reciben **el mismo `SKILL.md`** — el estándar abierto en acción — cada uno en su carpeta (`.claude/skills/`, `.cursor/skills/`); los *standards* (reglas siempre activas, que acá no usamos) viajan además al `AGENTS.md` de los agentes que lo leen. Cuando la skill cambie a v2, el equipo corre `packmind update` en cada repo y todos los agentes quedan al día — ese flujo de versionado es exactamente lo que cubre la [clase 3.6 L3](../../class_3_6_production/leccion3_versionamiento_skills/).

**¿Y self-hosted?** Packmind es open source ([github.com/PackmindHub/packmind](https://github.com/PackmindHub/packmind)): una organización que no quiere su playbook en la nube lo levanta con el docker-compose oficial y la misma CLI apunta a su instancia. Para la lección, el tier cloud gratuito basta.

## El fallback: qué hace Packmind por dentro

Sin cuenta de Packmind (o sin Node), el mecanismo de distribución se puede enseñar en ~20 líneas — porque *renderizar una skill para varios agentes* es, en el fondo, escribir el mismo contenido en las convenciones de carpeta de cada uno:

In [ ]:
def distribuir_skill(skill_md: Path, proyecto: Path) -> None:
    """Materializa un SKILL.md para tres agentes: Claude Code, Cursor y AGENTS.md."""
    skill = leer_skill(skill_md)
    nombre, descripcion, cuerpo = skill["name"], skill["description"], skill["cuerpo"]

    # Claude Code: la skill tal cual, en .claude/skills/<name>/SKILL.md
    destino = proyecto / ".claude" / "skills" / nombre / "SKILL.md"
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(skill_md.read_text())

    # Cursor: reglas en .cursor/rules/<name>.mdc con su propio frontmatter
    regla = proyecto / ".cursor" / "rules" / f"{nombre}.mdc"
    regla.parent.mkdir(parents=True, exist_ok=True)
    regla.write_text(f"---\ndescription: {descripcion}\n---\n\n{cuerpo}\n")

    # AGENTS.md: una sección por skill, apendizada
    agents = proyecto / "AGENTS.md"
    encabezado = "" if agents.exists() else "# AGENTS.md\n\nConocimiento operativo del proyecto.\n"
    with agents.open("a") as f:
        f.write(f"{encabezado}\n## Skill: {nombre}\n\n{cuerpo}\n")


CONSUMIDOR_LOCAL = Path("proyecto_consumidor_local")
if CONSUMIDOR_LOCAL.exists():
    shutil.rmtree(CONSUMIDOR_LOCAL)
CONSUMIDOR_LOCAL.mkdir()

for ruta in sorted(DIR_SKILLS.glob("*/SKILL.md")):
    distribuir_skill(ruta, CONSUMIDOR_LOCAL)

for archivo in sorted(p for p in CONSUMIDOR_LOCAL.rglob("*") if p.is_file()):
    print(f"  {archivo.relative_to(CONSUMIDOR_LOCAL)}")

Lo que el casero **no** hace es lo que justifica la herramienta real: playbook central con permisos, versionado y approval flow, detección de drift entre repos, y actualización en masa. La distribución es lo fácil; la *gobernanza* es el producto.

## El ecosistema, para seguir tirando del hilo

- **La especificación**: [agentskills.io](https://agentskills.io) — el estándar formal, con la lista de herramientas que lo soportan.
- **[anthropics/skills](https://github.com/anthropics/skills)**: skills oficiales curadas (crear .pptx, .xlsx, PDFs...) que puedes instalar tal cual — y leer como ejemplos de buena autoría.
- **[skills.sh](https://skills.sh)**: el marketplace comunitario más grande; miles de skills públicas, con lo bueno y lo malo que eso implica (lee lo que instalas: una skill son *instrucciones que tu agente va a seguir*).
- **Skills + MCP** (lección 1): una skill puede documentar *cómo usar bien* un servidor MCP — el conocimiento y la herramienta viajan por canales distintos y se encuentran en el agente.

La próxima vez que escribas dos veces el mismo párrafo de instrucciones para un agente, ya sabes: eso era una skill.